# 04 — Unified Model Comparison & Business Insights

**Objective:** Combine measured model metrics from Walmart, Amazon Reviews, and M5 into one tidy table, visualize comparisons within each task, and export a consolidated results file.

## Expected inputs
The three experiment notebooks save:
- `results/walmart_model_results.csv`
- `results/amazon_model_results.csv`
- `results/m5_model_results.csv`

This notebook never fabricates missing scores. It reports which result files are unavailable and only compares models using the same metric within a dataset.

In [ ]:
# 1. Imports and paths
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
result_files = [
    RESULTS_DIR / "walmart_model_results.csv",
    RESULTS_DIR / "amazon_model_results.csv",
    RESULTS_DIR / "m5_model_results.csv"
]

In [ ]:
# 2. Load available experiment results
frames, missing = [], []
for path in result_files:
    if path.exists():
        part = pd.read_csv(path)
        required = {"Dataset", "Model", "Metric", "Value"}
        if not required.issubset(part.columns):
            raise ValueError(f"{path} must contain {sorted(required)}")
        frames.append(part)
    else:
        missing.append(str(path))

if missing:
    print("Results not found yet (run the corresponding notebooks first):")
    for path in missing:
        print(" -", path)

if not frames:
    raise FileNotFoundError("No experiment result files found. Run notebooks 01–03 first.")

results = pd.concat(frames, ignore_index=True)
results["Value"] = pd.to_numeric(results["Value"], errors="coerce")
results = results.dropna(subset=["Value"])
display(results.sort_values(["Dataset", "Metric", "Value"]))

In [ ]:
# 3. Save unified results
unified_path = RESULTS_DIR / "unified_model_results.csv"
results.to_csv(unified_path, index=False)
print("Saved:", unified_path.resolve())

In [ ]:
# 4. Visualize one metric at a time, separately within each dataset
for (dataset, metric), group in results.groupby(["Dataset", "Metric"]):
    group = group.sort_values("Value", ascending=(metric in ["MAE", "RMSE", "Loss"]))
    plt.figure(figsize=(8, 4))
    plt.bar(group["Model"], group["Value"])
    plt.title(f"{dataset} — {metric}")
    plt.ylabel(metric)
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=.2)
    plt.tight_layout()
    plt.show()

In [ ]:
# 5. Identify best model per dataset/metric using sensible metric direction
lower_is_better = {"MAE", "RMSE", "MSE", "Loss", "Log Loss"}
summary = []
for (dataset, metric), group in results.groupby(["Dataset", "Metric"]):
    ascending = metric in lower_is_better
    winner = group.sort_values("Value", ascending=ascending).iloc[0]
    summary.append({
        "Dataset": dataset,
        "Metric": metric,
        "Selected_Model": winner["Model"],
        "Measured_Value": winner["Value"],
        "Selection_Rule": "Minimum" if ascending else "Maximum"
    })

best_table = pd.DataFrame(summary)
display(best_table)
best_table.to_csv(RESULTS_DIR / "best_model_summary.csv", index=False)

## Business interpretation template

Write conclusions only after inspecting your measured results. For each dataset, explain:

1. Which model performed best for the selected metric?
2. How large is the difference from the next-best model?
3. What operational decision could this support?
4. What are the limitations of the validation design?
5. What additional evidence is needed before deployment?

**Important:** Walmart/M5 forecasting metrics and Amazon classification metrics are not directly comparable. A lower RMSE does not mean a forecasting model is universally better than a sentiment classifier.

In [ ]:
# 6. Export a compact project result bundle
print("Unified results:", unified_path)
print("Best-model summary:", (RESULTS_DIR / "best_model_summary.csv").resolve())
print("Datasets represented:", sorted(results["Dataset"].unique()))
print("Models represented:", sorted(results["Model"].unique()))